In [8]:
# !pip install transformers datasets scikit-learn torch pandas numpy seaborn tqdm
# !pip install tqdm
# !pip install tiktoken
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s  0:00:00

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
from FakeBERT import FakeBERT
from FakeNewsDataset import FakeNewsDataset
from ParallelCNNHead import ParallelCNNHead

FakeBERT

FakeBERT.FakeBERT

In [3]:
"""
NLP Mini Project — FakeBERT + DeBERTa-v3 Improvement
=====================================================
Dataset : WELFake (saurabhshahane/fake-news-classification)
          Single file: WELFake_Dataset.csv
          Columns: Unnamed: 0 (index), title, text, label
          Label: 0 = Fake, 1 = Real  ← NOTE: opposite of paper

Upload the CSV to Colab:
  - Mount Drive OR use files.upload() OR place in /content/

Run cells top to bottom. GPU runtime recommended (Runtime > Change runtime type > T4 GPU).
"""

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install dependencies                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
# !pip install transformers tqdm -q

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Config                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer, BertModel,
    AutoTokenizer, AutoModel,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED       = 42
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 2e-5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load & Inspect Dataset                                ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Change this path if your file is somewhere else ──────────────
DATA_PATH = "WELFake_Dataset.csv"
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/WELFake_Dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:\n", df.head(3))
print("\nLabel distribution:\n", df["label"].value_counts())
print("\nNull values:\n", df.isnull().sum())

# WELFake label convention: 0 = Fake, 1 = Real
# We flip to match standard convention: 0 = Real, 1 = Fake
# This way label=1 always means "the positive class we want to detect"
df["label"] = df["label"].apply(lambda x: 0 if x == 1 else 1)
print("\nAfter flipping — 0=Real, 1=Fake:\n", df["label"].value_counts())


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Preprocessing                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

# Combine title + text (same as paper approach)
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()

# Drop any rows where content is empty after fill
df = df[df["content"].str.len() > 10].reset_index(drop=True)
print(f"Clean dataset size: {len(df)}")

# Train / test split — stratified
X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["label"],
    test_size=0.2, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Dataset Class                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

# class FakeNewsDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
#         self.texts     = texts.reset_index(drop=True)
#         self.labels    = labels.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_len   = max_len

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         enc = self.tokenizer(
#             self.texts[idx],
#             max_length=self.max_len,
#             padding="max_length",
#             truncation=True,
#             return_tensors="pt",
#         )
#         return {
#             "input_ids":      enc["input_ids"].squeeze(),
#             "attention_mask": enc["attention_mask"].squeeze(),
#             "label":          torch.tensor(self.labels[idx], dtype=torch.long),
#         }


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Model Architecture                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# class ParallelCNNHead(nn.Module):
    # """
    # 3 parallel 1D-CNN blocks (kernels 3,4,5) → concat → 2 more conv layers.
    # Matches FakeBERT Table 5 topology from the paper.
    # """
    # def __init__(self, hidden_size: int, num_filters: int = 128):
    #     super().__init__()
    #     self.conv3 = nn.Conv1d(hidden_size, num_filters, kernel_size=3, padding=1)
    #     self.conv4 = nn.Conv1d(hidden_size, num_filters, kernel_size=4, padding=2)
    #     self.conv5 = nn.Conv1d(hidden_size, num_filters, kernel_size=5, padding=2)
    #     self.pool  = nn.AdaptiveMaxPool1d(1)

    #     self.conv_post = nn.Conv1d(num_filters * 3, num_filters, kernel_size=5, padding=2)

    #     self.dense1  = nn.Linear(num_filters, 384)
    #     self.dense2  = nn.Linear(384, 128)
    #     self.dropout = nn.Dropout(0.2)
    #     self.relu    = nn.ReLU()

    # def forward(self, x):
    #     # x: (B, seq_len, hidden) → (B, hidden, seq_len)
    #     x = x.permute(0, 2, 1)

    #     b3 = self.pool(self.relu(self.conv3(x)))   # (B, 128, 1)
    #     b4 = self.pool(self.relu(self.conv4(x)))
    #     b5 = self.pool(self.relu(self.conv5(x)))

    #     cat = torch.cat([b3, b4, b5], dim=1)       # (B, 384, 1)
    #     cat = cat.expand(-1, -1, 10)               # expand seq dim for post-conv
    #     out = self.pool(self.relu(self.conv_post(cat)))  # (B, 128, 1)
    #     out = out.squeeze(-1)                       # (B, 128)

    #     out = self.dropout(self.relu(self.dense1(out)))
    #     out = self.dropout(self.relu(self.dense2(out)))
    #     return out


# class FakeBERT(nn.Module):
#     """Encoder-agnostic — swap BERT for DeBERTa by passing a different encoder."""
#     def __init__(self, encoder, hidden_size: int, num_classes: int = 2):
#         super().__init__()
#         self.encoder    = encoder
#         self.cnn_head   = ParallelCNNHead(hidden_size)
#         self.classifier = nn.Linear(128, num_classes)

#     def forward(self, input_ids, attention_mask):
#         out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
#         seq_out  = out.last_hidden_state          # (B, seq_len, hidden)
#         features = self.cnn_head(seq_out)         # (B, 128)
#         logits   = self.classifier(features)      # (B, 2)
#         return logits


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Training & Evaluation Utilities                      ║
# ╚══════════════════════════════════════════════════════════════════╝

def train_epoch(model, loader, optimizer, criterion, epoch, model_name):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    pbar = tqdm(loader, desc=f"[{model_name}] Epoch {epoch} Train", leave=False, unit="batch")
    cnt = 0
    for batch in pbar:
        cnt += 1
        if cnt % 100 ==0:
            print(f"{cnt} batches done\n")
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=1).cpu().tolist()
        all_labels  += lbls.cpu().tolist()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def evaluate(model, loader, criterion, desc="Evaluating"):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []
    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    for batch in pbar:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)

        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().tolist()

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=1).cpu().tolist()
        all_labels  += lbls.cpu().tolist()
        all_probs   += probs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), all_labels, all_preds, all_probs


def compute_metrics(labels, preds, probs, model_name):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)
    cm   = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr_val = fp / (fp + tn)
    fnr_val = fn / (fn + tp)
    fpr_arr, tpr_arr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr_arr, tpr_arr)

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  FPR       : {fpr_val:.4f}")
    print(f"  FNR       : {fnr_val:.4f}")
    print(f"  AUC-ROC   : {roc_auc:.4f}")
    print(classification_report(labels, preds, target_names=["Real", "Fake"]))

    return dict(model=model_name, accuracy=acc, precision=prec, recall=rec,
                f1=f1, fpr=fpr_val, fnr=fnr_val, auc=roc_auc,
                cm=cm, fpr_arr=fpr_arr, tpr_arr=tpr_arr)


def run_training(model, train_loader, val_loader, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.1, total_iters=EPOCHS
    )
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    epoch_pbar = tqdm(range(1, EPOCHS + 1), desc=f"{model_name}", unit="epoch")
    for epoch in epoch_pbar:
        tr_loss, tr_acc          = train_epoch(model, train_loader, optimizer, criterion, epoch, model_name)
        vl_loss, lbls, preds, _ = evaluate(model, val_loader, criterion,
                                            desc=f"[{model_name}] Epoch {epoch} Val")
        vl_acc = accuracy_score(lbls, preds)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        epoch_pbar.set_postfix(
            tr_loss=f"{tr_loss:.4f}", tr_acc=f"{tr_acc:.4f}",
            vl_loss=f"{vl_loss:.4f}", vl_acc=f"{vl_acc:.4f}"
        )

    return history


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Visualisations                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

def plot_all(histories, results):
    colors = {"FakeBERT (BERT-base)": "steelblue", "FakeBERT (DeBERTa-v3)": "tomato"}

    # — Training curves —
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in histories.items():
        c = colors[name]
        axes[0].plot(h["train_loss"], "--", color=c, label=f"{name} Train")
        axes[0].plot(h["val_loss"],         color=c, label=f"{name} Val")
        axes[1].plot(h["train_acc"], "--",  color=c, label=f"{name} Train")
        axes[1].plot(h["val_acc"],          color=c, label=f"{name} Val")
    for ax, title in zip(axes, ["Cross-Entropy Loss", "Accuracy"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Training Curves"); plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150); plt.show()

    # — Confusion matrices —
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, res in zip(axes, results):
        sns.heatmap(res["cm"], annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Real","Fake"], yticklabels=["Real","Fake"], ax=ax)
        ax.set_title(res["model"]); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150); plt.show()

    # — ROC curves —
    plt.figure(figsize=(8, 6))
    for res, c in zip(results, ["steelblue","tomato"]):
        plt.plot(res["fpr_arr"], res["tpr_arr"], color=c,
                 label=f"{res['model']} (AUC={res['auc']:.4f})")
    plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve"); plt.legend(); plt.grid(alpha=0.3)
    plt.savefig("roc_curves.png", dpi=150); plt.show()

    # — Metric comparison bar chart —
    metrics = ["accuracy","precision","recall","f1","auc"]
    x, w = np.arange(len(metrics)), 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (res, c) in enumerate(zip(results, ["steelblue","tomato"])):
        vals = [res[m] for m in metrics]
        bars = ax.bar(x + i*w, vals, w, label=res["model"], color=c, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f"{v:.3f}", ha="center", fontsize=8)
    ax.set_xticks(x + w/2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylim(0.85, 1.02); ax.set_ylabel("Score")
    ax.set_title("Model Comparison"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("metric_comparison.png", dpi=150); plt.show()


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — MODEL A: FakeBERT + BERT-base (Paper Baseline)       ║
# ╚══════════════════════════════════════════════════════════════════╝

print("\n" + "="*55)
print("  MODEL A: FakeBERT + BERT-base-uncased  [Paper Baseline]")
print("="*55)

bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
bert_enc = BertModel.from_pretrained("bert-base-uncased")

train_ds_b  = FakeNewsDataset(X_train, y_train, bert_tok)
test_ds_b   = FakeNewsDataset(X_test,  y_test,  bert_tok)
train_dl_b  = DataLoader(train_ds_b, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl_b   = DataLoader(test_ds_b,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model_bert  = FakeBERT(bert_enc, hidden_size=768).to(DEVICE)
hist_bert   = run_training(model_bert, train_dl_b, test_dl_b, "FakeBERT (BERT-base)")

_, lbls, preds, probs = evaluate(model_bert, test_dl_b, nn.CrossEntropyLoss(), desc="[BERT-base] Final Test Eval")
res_bert = compute_metrics(lbls, preds, probs, "FakeBERT (BERT-base)")

torch.save(model_bert.state_dict(), "fakebert_bert.pt")
print("Model A saved.")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MODEL B: FakeBERT + DeBERTa-v3 (Your Improvement)  ║
# ╚══════════════════════════════════════════════════════════════════╝

print("\n" + "="*55)
print("  MODEL B: FakeBERT + DeBERTa-v3-base  [Improvement]")
print("="*55)

deb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
deb_enc = AutoModel.from_pretrained("microsoft/deberta-v3-base")

train_ds_d  = FakeNewsDataset(X_train, y_train, deb_tok)
test_ds_d   = FakeNewsDataset(X_test,  y_test,  deb_tok)
train_dl_d  = DataLoader(train_ds_d, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl_d   = DataLoader(test_ds_d,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model_deb   = FakeBERT(deb_enc, hidden_size=768).to(DEVICE)
hist_deb    = run_training(model_deb, train_dl_d, test_dl_d, "FakeBERT (DeBERTa-v3)")

_, lbls, preds, probs = evaluate(model_deb, test_dl_d, nn.CrossEntropyLoss(), desc="[DeBERTa-v3] Final Test Eval")
res_deb = compute_metrics(lbls, preds, probs, "FakeBERT (DeBERTa-v3)")

torch.save(model_deb.state_dict(), "fakebert_deberta.pt")
print("Model B saved.")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Plots + Summary Table                               ║
# ╚══════════════════════════════════════════════════════════════════╝

histories   = {"FakeBERT (BERT-base)": hist_bert, "FakeBERT (DeBERTa-v3)": hist_deb}
results     = [res_bert, res_deb]

plot_all(histories, results)

summary = pd.DataFrame([{
    "Model":     r["model"],
    "Accuracy":  f"{r['accuracy']:.4f}",
    "Precision": f"{r['precision']:.4f}",
    "Recall":    f"{r['recall']:.4f}",
    "F1":        f"{r['f1']:.4f}",
    "FPR":       f"{r['fpr']:.4f}",
    "FNR":       f"{r['fnr']:.4f}",
    "AUC-ROC":   f"{r['auc']:.4f}",
} for r in results])

print("\n", summary.to_string(index=False))
summary.to_csv("results_summary.csv", index=False)
print("\nAll plots and results_summary.csv saved to /content/")

/Users/aaryanurunkar/Desktop/Everything/ml/NLP_proj/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
Shape: (72134, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']

First 3 rows:
    Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  

Label distribution:
 label
1    37106
0    35028
Name: count, dtype: int64

Null values:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After flipping — 0=Real, 1=Fake:
 label
0    37106
1    35028
Name: count, dtype: int64
Clean dataset size: 72119
Train: 57695 | Test: 14424

  MODEL A: FakeBERT + BERT-base-uncased  [Paper Baseline]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19294.19it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
FakeBERT (BERT-base):   0%|          | 0/5 [00:00<?, ?epoch/s]

100 batches done



200 batches done



...

1800 batches done



FakeBERT (BERT-base): 100%|██████████| 5/5 [1:49:39<00:00, 1315.81s/epoch, tr_acc=0.9997, tr_loss=0.0013, vl_acc=0.9942, vl_loss=0.0472]



  FakeBERT (BERT-base)
  Accuracy  : 0.9942
  Precision : 0.9943
  Recall    : 0.9939
  F1-Score  : 0.9941
  FPR       : 0.0054
  FNR       : 0.0061
  AUC-ROC   : 0.9996
              precision    recall  f1-score   support

        Real       0.99      0.99      0.99      7418
        Fake       0.99      0.99      0.99      7006

    accuracy                           0.99     14424
   macro avg       0.99      0.99      0.99     14424
weighted avg       0.99      0.99      0.99     14424

Model A saved.

  MODEL B: FakeBERT + DeBERTa-v3-base  [Improvement]


[transformers] Could not extract SentencePiece model from /Users/aaryanurunkar/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.

In [2]:
"""
NLP Mini Project — FakeBERT + DeBERTa-v3 Improvement
=====================================================
Dataset : WELFake (saurabhshahane/fake-news-classification)
          Single file: WELFake_Dataset.csv
          Columns: Unnamed: 0 (index), title, text, label
          Label: 0 = Fake, 1 = Real  ← NOTE: opposite of paper

Upload the CSV to Colab:
  - Mount Drive OR use files.upload() OR place in /content/

Run cells top to bottom. GPU runtime recommended (Runtime > Change runtime type > T4 GPU).
"""

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install dependencies                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
# !pip install transformers tqdm -q

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Config                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer, BertModel,
    AutoTokenizer, AutoModel,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED       = 42
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 2e-5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load & Inspect Dataset                                ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Change this path if your file is somewhere else ──────────────
DATA_PATH = "WELFake_Dataset.csv"
# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/WELFake_Dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:\n", df.head(3))
print("\nLabel distribution:\n", df["label"].value_counts())
print("\nNull values:\n", df.isnull().sum())

# WELFake label convention: 0 = Fake, 1 = Real
# We flip to match standard convention: 0 = Real, 1 = Fake
# This way label=1 always means "the positive class we want to detect"
df["label"] = df["label"].apply(lambda x: 0 if x == 1 else 1)
print("\nAfter flipping — 0=Real, 1=Fake:\n", df["label"].value_counts())


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Preprocessing                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

# Combine title + text (same as paper approach)
df["content"] = (df["title"].fillna("") + " " + df["text"].fillna("")).str.strip()

# Drop any rows where content is empty after fill
df = df[df["content"].str.len() > 10].reset_index(drop=True)
print(f"Clean dataset size: {len(df)}")

# Train / test split — stratified
X_train, X_test, y_train, y_test = train_test_split(
    df["content"], df["label"],
    test_size=0.2, stratify=df["label"], random_state=SEED
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Dataset Class                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

# class FakeNewsDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
#         self.texts     = texts.reset_index(drop=True)
#         self.labels    = labels.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_len   = max_len

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         enc = self.tokenizer(
#             self.texts[idx],
#             max_length=self.max_len,
#             padding="max_length",
#             truncation=True,
#             return_tensors="pt",
#         )
#         return {
#             "input_ids":      enc["input_ids"].squeeze(),
#             "attention_mask": enc["attention_mask"].squeeze(),
#             "label":          torch.tensor(self.labels[idx], dtype=torch.long),
#         }


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Model Architecture                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# class ParallelCNNHead(nn.Module):
    # """
    # 3 parallel 1D-CNN blocks (kernels 3,4,5) → concat → 2 more conv layers.
    # Matches FakeBERT Table 5 topology from the paper.
    # """
    # def __init__(self, hidden_size: int, num_filters: int = 128):
    #     super().__init__()
    #     self.conv3 = nn.Conv1d(hidden_size, num_filters, kernel_size=3, padding=1)
    #     self.conv4 = nn.Conv1d(hidden_size, num_filters, kernel_size=4, padding=2)
    #     self.conv5 = nn.Conv1d(hidden_size, num_filters, kernel_size=5, padding=2)
    #     self.pool  = nn.AdaptiveMaxPool1d(1)

    #     self.conv_post = nn.Conv1d(num_filters * 3, num_filters, kernel_size=5, padding=2)

    #     self.dense1  = nn.Linear(num_filters, 384)
    #     self.dense2  = nn.Linear(384, 128)
    #     self.dropout = nn.Dropout(0.2)
    #     self.relu    = nn.ReLU()

    # def forward(self, x):
    #     # x: (B, seq_len, hidden) → (B, hidden, seq_len)
    #     x = x.permute(0, 2, 1)

    #     b3 = self.pool(self.relu(self.conv3(x)))   # (B, 128, 1)
    #     b4 = self.pool(self.relu(self.conv4(x)))
    #     b5 = self.pool(self.relu(self.conv5(x)))

    #     cat = torch.cat([b3, b4, b5], dim=1)       # (B, 384, 1)
    #     cat = cat.expand(-1, -1, 10)               # expand seq dim for post-conv
    #     out = self.pool(self.relu(self.conv_post(cat)))  # (B, 128, 1)
    #     out = out.squeeze(-1)                       # (B, 128)

    #     out = self.dropout(self.relu(self.dense1(out)))
    #     out = self.dropout(self.relu(self.dense2(out)))
    #     return out


# class FakeBERT(nn.Module):
#     """Encoder-agnostic — swap BERT for DeBERTa by passing a different encoder."""
#     def __init__(self, encoder, hidden_size: int, num_classes: int = 2):
#         super().__init__()
#         self.encoder    = encoder
#         self.cnn_head   = ParallelCNNHead(hidden_size)
#         self.classifier = nn.Linear(128, num_classes)

#     def forward(self, input_ids, attention_mask):
#         out      = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
#         seq_out  = out.last_hidden_state          # (B, seq_len, hidden)
#         features = self.cnn_head(seq_out)         # (B, 128)
#         logits   = self.classifier(features)      # (B, 2)
#         return logits


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Training & Evaluation Utilities                      ║
# ╚══════════════════════════════════════════════════════════════════╝

def train_epoch(model, loader, optimizer, criterion, epoch, model_name):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    pbar = tqdm(loader, desc=f"[{model_name}] Epoch {epoch} Train", leave=False, unit="batch")
    cnt = 0
    for batch in pbar:
        cnt += 1
        if cnt % 100 ==0:
            print(f"{cnt} batches done\n")
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=1).cpu().tolist()
        all_labels  += lbls.cpu().tolist()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def evaluate(model, loader, criterion, desc="Evaluating"):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []
    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    for batch in pbar:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)

        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().tolist()

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=1).cpu().tolist()
        all_labels  += lbls.cpu().tolist()
        all_probs   += probs

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), all_labels, all_preds, all_probs


def compute_metrics(labels, preds, probs, model_name):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)
    cm   = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr_val = fp / (fp + tn)
    fnr_val = fn / (fn + tp)
    fpr_arr, tpr_arr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr_arr, tpr_arr)

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  FPR       : {fpr_val:.4f}")
    print(f"  FNR       : {fnr_val:.4f}")
    print(f"  AUC-ROC   : {roc_auc:.4f}")
    print(classification_report(labels, preds, target_names=["Real", "Fake"]))

    return dict(model=model_name, accuracy=acc, precision=prec, recall=rec,
                f1=f1, fpr=fpr_val, fnr=fnr_val, auc=roc_auc,
                cm=cm, fpr_arr=fpr_arr, tpr_arr=tpr_arr)


def run_training(model, train_loader, val_loader, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0, end_factor=0.1, total_iters=EPOCHS
    )
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    epoch_pbar = tqdm(range(1, EPOCHS + 1), desc=f"{model_name}", unit="epoch")
    for epoch in epoch_pbar:
        tr_loss, tr_acc          = train_epoch(model, train_loader, optimizer, criterion, epoch, model_name)
        vl_loss, lbls, preds, _ = evaluate(model, val_loader, criterion,
                                            desc=f"[{model_name}] Epoch {epoch} Val")
        vl_acc = accuracy_score(lbls, preds)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        epoch_pbar.set_postfix(
            tr_loss=f"{tr_loss:.4f}", tr_acc=f"{tr_acc:.4f}",
            vl_loss=f"{vl_loss:.4f}", vl_acc=f"{vl_acc:.4f}"
        )

    return history


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Visualisations                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

def plot_all(histories, results):
    colors = {"FakeBERT (BERT-base)": "steelblue", "FakeBERT (DeBERTa-v3)": "tomato"}

    # — Training curves —
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, h in histories.items():
        c = colors[name]
        axes[0].plot(h["train_loss"], "--", color=c, label=f"{name} Train")
        axes[0].plot(h["val_loss"],         color=c, label=f"{name} Val")
        axes[1].plot(h["train_acc"], "--",  color=c, label=f"{name} Train")
        axes[1].plot(h["val_acc"],          color=c, label=f"{name} Val")
    for ax, title in zip(axes, ["Cross-Entropy Loss", "Accuracy"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Training Curves"); plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150); plt.show()

    # — Confusion matrices —
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, res in zip(axes, results):
        sns.heatmap(res["cm"], annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Real","Fake"], yticklabels=["Real","Fake"], ax=ax)
        ax.set_title(res["model"]); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150); plt.show()

    # — ROC curves —
    plt.figure(figsize=(8, 6))
    for res, c in zip(results, ["steelblue","tomato"]):
        plt.plot(res["fpr_arr"], res["tpr_arr"], color=c,
                 label=f"{res['model']} (AUC={res['auc']:.4f})")
    plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve"); plt.legend(); plt.grid(alpha=0.3)
    plt.savefig("roc_curves.png", dpi=150); plt.show()

    # — Metric comparison bar chart —
    metrics = ["accuracy","precision","recall","f1","auc"]
    x, w = np.arange(len(metrics)), 0.35
    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (res, c) in enumerate(zip(results, ["steelblue","tomato"])):
        vals = [res[m] for m in metrics]
        bars = ax.bar(x + i*w, vals, w, label=res["model"], color=c, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f"{v:.3f}", ha="center", fontsize=8)
    ax.set_xticks(x + w/2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylim(0.85, 1.02); ax.set_ylabel("Score")
    ax.set_title("Model Comparison"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("metric_comparison.png", dpi=150); plt.show()


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — MODEL A: FakeBERT + BERT-base (Paper Baseline)       ║
# ╚══════════════════════════════════════════════════════════════════╝

# print("\n" + "="*55)
# print("  MODEL A: FakeBERT + BERT-base-uncased  [Paper Baseline]")
# print("="*55)

# bert_tok = BertTokenizer.from_pretrained("bert-base-uncased")
# bert_enc = BertModel.from_pretrained("bert-base-uncased")

# train_ds_b  = FakeNewsDataset(X_train, y_train, bert_tok)
# test_ds_b   = FakeNewsDataset(X_test,  y_test,  bert_tok)
# train_dl_b  = DataLoader(train_ds_b, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
# test_dl_b   = DataLoader(test_ds_b,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# model_bert  = FakeBERT(bert_enc, hidden_size=768).to(DEVICE)
# hist_bert   = run_training(model_bert, train_dl_b, test_dl_b, "FakeBERT (BERT-base)")

# _, lbls, preds, probs = evaluate(model_bert, test_dl_b, nn.CrossEntropyLoss(), desc="[BERT-base] Final Test Eval")
# res_bert = compute_metrics(lbls, preds, probs, "FakeBERT (BERT-base)")

# torch.save(model_bert.state_dict(), "fakebert_bert.pt")
# print("Model A saved.")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MODEL B: FakeBERT + DeBERTa-v3 (Your Improvement)  ║
# ╚══════════════════════════════════════════════════════════════════╝

print("\n" + "="*55)
print("  MODEL B: FakeBERT + DeBERTa-v3-base  [Improvement]")
print("="*55)

deb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
deb_enc = AutoModel.from_pretrained("microsoft/deberta-v3-base")

train_ds_d  = FakeNewsDataset(X_train, y_train, deb_tok)
test_ds_d   = FakeNewsDataset(X_test,  y_test,  deb_tok)
train_dl_d  = DataLoader(train_ds_d, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl_d   = DataLoader(test_ds_d,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model_deb   = FakeBERT(deb_enc, hidden_size=768).to(DEVICE)

model_deb = model_deb.float()

hist_deb    = run_training(model_deb, train_dl_d, test_dl_d, "FakeBERT (DeBERTa-v3)")

_, lbls, preds, probs = evaluate(model_deb, test_dl_d, nn.CrossEntropyLoss(), desc="[DeBERTa-v3] Final Test Eval")
res_deb = compute_metrics(lbls, preds, probs, "FakeBERT (DeBERTa-v3)")

torch.save(model_deb.state_dict(), "fakebert_deberta.pt")
print("Model B saved.")


# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Plots + Summary Table                               ║
# ╚══════════════════════════════════════════════════════════════════╝

histories   = {"FakeBERT (BERT-base)": hist_bert, "FakeBERT (DeBERTa-v3)": hist_deb}
results     = [res_bert, res_deb]

plot_all(histories, results)

summary = pd.DataFrame([{
    "Model":     r["model"],
    "Accuracy":  f"{r['accuracy']:.4f}",
    "Precision": f"{r['precision']:.4f}",
    "Recall":    f"{r['recall']:.4f}",
    "F1":        f"{r['f1']:.4f}",
    "FPR":       f"{r['fpr']:.4f}",
    "FNR":       f"{r['fnr']:.4f}",
    "AUC-ROC":   f"{r['auc']:.4f}",
} for r in results])

print("\n", summary.to_string(index=False))
summary.to_csv("results_summary.csv", index=False)
print("\nAll plots and results_summary.csv saved to /content/")

/Users/aaryanurunkar/Desktop/Everything/ml/NLP_proj/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
Shape: (72134, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']

First 3 rows:
    Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  

Label distribution:
 label
1    37106
0    35028
Name: count, dtype: int64

Null values:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After flipping — 0=Real, 1=Fake:
 label
0    37106
1    35028
Name: count, dtype: int64
Clean dataset size: 72119
Train: 57695 | Test: 14424

  MODEL B: FakeBERT + DeBERTa-v3-base  [Improvement]


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 50993.01it/s]
[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

100 batches done



200 batches done



...

1700 batches done



1800 batches done



FakeBERT (DeBERTa-v3): 100%|██████████| 5/5 [3:05:36<00:00, 2227.39s/epoch, tr_acc=0.9998, tr_loss=0.0012, vl_acc=0.9974, vl_loss=0.0181]



  FakeBERT (DeBERTa-v3)
  Accuracy  : 0.9974
  Precision : 0.9966
  Recall    : 0.9981
  F1-Score  : 0.9974
  FPR       : 0.0032
  FNR       : 0.0019
  AUC-ROC   : 0.9998
              precision    recall  f1-score   support

        Real       1.00      1.00      1.00      7418
        Fake       1.00      1.00      1.00      7006

    accuracy                           1.00     14424
   macro avg       1.00      1.00      1.00     14424
weighted avg       1.00      1.00      1.00     14424

Model B saved.


NameError: name 'hist_bert' is not defined

In [2]:
!pip install protobuf


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
